In [1]:
import torch
from torch.utils.data import DataLoader, Dataset

In [2]:
class MNISTDataset(Dataset):
    def __init__(self, file_path):
        self.images, self.labels = self._read_file(file_path)

    def _read_file(self, file_path):
        images = []
        labels = []
        with open(file_path, 'r') as f:
            next(f)  #skip 1st row
            for line in f:
                line = line.rstrip('\n')  #discard the last \n in the end
                items = line.split(',')  #split 
                images.append([float(x) for x in items[1:]])  #from 2 till end
                labels.append(int(items[0]))
            return images, labels

    def __getitem__(self, index):
        image, label = self.images[index], self.labels[index]
        image = torch.tensor(image)
        image = image/255
        image = (image - 0.1307) / 0.3081  #calculated before based on the dataset
        label = torch.tensor(label)
        return image, label

    def __len__(self):
        return len(self.images)

In [3]:
batch_size = 64
train_dataset = MNISTDataset(r'C:\work\VScode\vscodeproject\AI\PyTorch\MNIST\mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = MNISTDataset(r'C:\work\VScode\vscodeproject\AI\PyTorch\MNIST\mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [4]:
learning_rate = 0.1
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
layer_sizes = [28*28, 128, 128, 128, 64, 10]  #NN shape
weights = []
biases = []
#match the pairs to map (784,128), (128,128), (128,128), (128,64), (64,10). the layer to transfer: 28*28 to 128...
for in_size, out_size in zip(layer_sizes[:-1], layer_sizes[1:]):
    w = torch.randn(in_size, out_size, device=device)*torch.sqrt(torch.tensor(2/in_size))
    b = torch.zeros(out_size, device=device)
    weights.append(w)
    biases.append(b)

In [5]:
#activation function
def relu(x):
    return torch.clamp(x, min=0)

def relu_gradient(x):
    return (x > 0).float()

def softmax(x):
    x_exp = torch.exp(x - torch.max(x, dim=1, keepdim=True)[0])  #for numerical stability
    return x_exp / torch.sum(x_exp, dim=1, keepdim=True)

#BCELoss
def cross_entropy_loss(pred, labels):
    N = pred.shape[0]
    log_likelihood = -torch.log(pred[range(N), labels])
    loss = torch.sum(log_likelihood) / N
    return loss

In [ ]:
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        #forward pass
        activations = [images]
        for w, b in zip(weights[:-1], biases[:-1]):
            z = activations[-1] @ w + b
            a = relu(z)
            activations.append(a)
        z = activations[-1] @ weights[-1] + biases[-1]
        pred = softmax(z)

        loss = cross_entropy_loss(pred, labels)
        total_loss += loss.item()

        #backward pass
        grad_pred = pred.clone()
        grad_pred[range(grad_pred.shape[0]), labels] -= 1
        grad_pred /= grad_pred.shape[0]

        grad_w = []
        grad_b = []
        grad_a = grad_pred
        for i in reversed(range(len(weights))):
            grad_w_i = activations[i].T @ grad_a
            grad_b_i = torch.sum(grad_a, dim=0)
            grad_w.insert(0, grad_w_i)
            grad_b.insert(0, grad_b_i)

            if i > 0:
                grad_a = (grad_a @ weights[i].T) * relu_gradient(activations[i])

        #update
        for i in range(len(weights)):
            weights[i] -= learning_rate * grad_w[i]
            biases[i] -= learning_rate * grad_b[i]

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}')

Epoch 1/10, Loss: 0.0007
Epoch 2/10, Loss: 0.0002
Epoch 3/10, Loss: 0.0001
Epoch 4/10, Loss: 0.0001
Epoch 5/10, Loss: 0.0001
Epoch 6/10, Loss: 0.0001
Epoch 7/10, Loss: 0.0001
Epoch 8/10, Loss: 0.0001
Epoch 9/10, Loss: 0.0001
Epoch 10/10, Loss: 0.0000


In [11]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        activations = [images]
        for w, b in zip(weights[:-1], biases[:-1]):
            z = activations[-1] @ w + b
            a = relu(z)
            activations.append(a)
        z = activations[-1] @ weights[-1] + biases[-1]
        pred = softmax(z)
        predicted_labels = torch.argmax(pred, dim=1)
        correct += (predicted_labels == labels).sum().item()
        total += labels.size(0)

    print(f'Accuracy: {correct/total:.4f}')

Accuracy: 0.9821
